<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Ranking measures

**[Ranking measures](https://learning.nextia-ai.com/courses/search/m02/ranking-measures/)** · Search and Ranking: Find the Best Match · Module 2, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** compute precision@k, recall@k, the reciprocal rank and NDCG by hand for one query, check them against the course's helper, and measure many queries at once.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (about 10 MB). No account, no key and no model download. |
| **You should know** | Graded judgments, from [Relevance judgments](https://learning.nextia-ai.com/courses/search/m02/relevance-judgments/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m02/ranking-measures/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M02-L02-ranking-measures/ranking-measures-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 10 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/catalogue.csv.gz": "569c1c4c41403d59492e4cb22be61a7e634b052f67b4d32d12a9f14e77c08c52",
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/judgments.csv.gz": "6a0b896c64a0e1000fe9401bfc359ae7995732af458120963408d71e50333ec6",
    "data/recorded/rerank.csv.gz": "38971117046aefd883c159ed9a7b196a0ca88f54757a8dc35df956d39222da88",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One query, ten results

The lesson works one query by hand: "3 piece rug set with runners". The list is BM25's top 10, and the grades are the real WANDS judgments: 2 exact, 1 partial, 0 irrelevant. Run the next cell.

In [ ]:
j = judgments()
qid = "W-236"           # "3 piece rug set with runners", a validation query
ranked = ['LK-67020', 'LK-52790', 'LK-47233', 'LK-48700', 'LK-28644', 'LK-11071', 'LK-83085', 'LK-22935', 'LK-29462', 'LK-55118']   # BM25's top 10
judged = j[qid]
grades = [judged.get(p, 0) for p in ranked]
print(grades)
print("exact products for this query:", sum(1 for g in judged.values() if g == 2))
show(ranked, judged)

> **Check.** The grades are 2, 1, 2, 2, 0, 2, 1, 2, 2, 1. The bath set at rank 5 is irrelevant. The query has 28 exact products in all, so ten results can never find all of them.

## 2. Precision, recall and the reciprocal rank

Three simple measures count only the **exact** products:

- **precision@k**: the share of the top k that is exact;
- **recall@k**: the share of all exact products that the top k finds;
- **reciprocal rank**: 1 divided by the rank of the first exact product. Its mean over many queries is **MRR**.

In [ ]:
def precision_at(ranked, judged, k):
    """The share of the top k that is exact."""
    return sum(1 for p in ranked[:k] if judged.get(p, 0) == 2) / k

def recall_at(ranked, judged, k):
    """The share of all exact products that the top k finds."""
    exact = [p for p, g in judged.items() if g == 2]
    return sum(1 for p in ranked[:k] if judged.get(p, 0) == 2) / len(exact)

def reciprocal_rank(ranked, judged):
    """1 / the rank of the first exact product (0 if there is none)."""
    for rank, p in enumerate(ranked, 1):
        if judged.get(p, 0) == 2:
            return 1 / rank
    return 0.0

print("precision@5", precision_at(ranked, judged, 5), " precision@10", precision_at(ranked, judged, 10))
print("recall@10", round(recall_at(ranked, judged, 10), 3), " reciprocal rank", reciprocal_rank(ranked, judged))

> **Check.** Precision@5 is 0.6 (3 of 5) and precision@10 is 0.6 (6 of 10). Recall@10 is 0.214 (6 of 28). The first result is exact, so the reciprocal rank is 1.

## 3. Your turn: NDCG

**NDCG** (normalised discounted cumulative gain) uses every grade, and it cares about the order. Each result adds its **gain** (2**grade − 1: exact 3, partial 1, irrelevant 0) divided by a **discount** that grows with the rank (log2(rank + 1)). The sum is **DCG**. Divide it by the DCG of the best possible list, and you get a number from 0 to 1. Complete `dcg_at`.

In [ ]:
import math

def dcg_at(grades, k):
    """Add gain / discount for the first k grades: gain = 2**grade - 1, discount = log2(rank + 1)."""
    return ...   # sum over rank = 1, 2, ... k of (2**grade - 1) / math.log2(rank + 1)

def ndcg_at(ranked, judged, k):
    """DCG of the list divided by the DCG of the best possible list (from every judged product)."""
    grades = [judged.get(p, 0) for p in ranked]
    ideal = sorted(judged.values(), reverse=True)
    if dcg_at(grades, k) is ...:
        return None
    return dcg_at(grades, k) / dcg_at(ideal, k)

ndcg10 = ndcg_at(ranked, judged, 10)
print(ndcg10, " compare.py:", round(cp.ndcg(ranked, judged, 10), 3))

In [ ]:
expect('ndcg10', ndcg10, 0.731, tolerance=0.001)

## 4. One move

> **Predict.** Move the irrelevant bath set from rank 5 to rank 1. Precision@10 does not change: the same ten products are there. How much does NDCG@10 fall?

In [ ]:
moved = [ranked[4]] + ranked[:4] + ranked[5:]     # the bath set (rank 5) moves to rank 1
moved_ndcg10 = cp.ndcg(moved, judged, 10)
print("NDCG@10:", round(cp.ndcg(ranked, judged, 10), 3), "->", round(moved_ndcg10, 3))
print("precision@10:", precision_at(ranked, judged, 10), "->", precision_at(moved, judged, 10))

> **Check.** NDCG@10 falls from 0.731 to 0.615, and precision@10 stays at 0.6. NDCG sees the order; precision@10 does not.

## 5. Many queries

One query is an example, not a measure. Run the same measures on every validation query. The ranking here is the **hybrid** search that you build in Module 5, recorded for every validation and test query, so this notebook needs no search code. `cp.per_query` gives one row per query.

In [ ]:
q = queries().set_index("query_id")
rr = pd.read_csv("data/recorded/rerank.csv.gz")           # query_id, product_id, hybrid_rank, score
run = {qid: list(g.sort_values("hybrid_rank")["product_id"]) for qid, g in rr.groupby("query_id")}
val = {qid: r for qid, r in run.items() if q.loc[qid, "split"] == "validation"}
pq = cp.per_query(val, j)                                  # one row per query, every measure
print(len(pq), "validation queries")
pq[["ndcg@10", "precision@5", "recall@100", "mrr"]].mean().round(3)

> **Check.** 121 queries. Mean NDCG@10 0.621, MRR 0.584, recall@100 0.563. Each number answers a different question about the same lists.

## 6. Your turn: which measure for which search?

In [ ]:
pq["kind"] = q.loc[pq.index, "qtype"]
by_kind = pq.groupby("kind")[["ndcg@10", "mrr", "recall@100"]].mean().round(3)
by_kind

In [ ]:
best_mrr_kind = ...   # the kind of query with the highest MRR in the table above
print(best_mrr_kind)

In [ ]:
expect('best_mrr_kind', best_mrr_kind, 'known-item')

> **Check.** **known-item** queries have the highest MRR (0.67). A known-item search ("kisner coffee table") has one right answer, so the rank of the first exact product is what the person feels: MRR fits it. A broad search ("sofa") has many good answers, so NDCG@10 fits it better. Recall@100 matters for the first stage of a two-stage search (Module 6).

## 7. In the project

```sh
search eval --run bm25              # NDCG@10 with its 95% interval, recall@100, MRR
search eval --run bm25 --by qtype   # the same by kind of query
```

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Build a test set that represents your users](https://learning.nextia-ai.com/courses/search/m02/build-a-test-set-that-represents-your-users/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m02/ranking-measures/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.